# 08. Error analysis (distilled student, internal test)

For entities with the correct span but wrong type: error kinds, most frequent confusions, and attribution of each
error to its likely source (inherited teacher convention, unseen word, mixed teacher labels, or genuine student error).
Post-hoc analysis only: not used to tune the system.

In [ ]:
import os, json, re
from collections import Counter
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if not os.path.exists("/content/maintie"):
    os.system("git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie")
BASE = "/content/drive/MyDrive/mistral_ft"; RUN_NAME = "distilled_qwen_nothink_v1"
gold = json.load(open("/content/maintie/data/gold_release.json"))
preds = [json.loads(l) for l in open(f"{BASE}/results/predictions_{RUN_NAME}.jsonl") if l.strip()]
train_rows = [json.loads(l) for l in open(f"{BASE}/data/train.jsonl") if l.strip()]

def parse_json(raw):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (raw or "").strip())
    s, e = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[s:e + 1]) if s != -1 and e != -1 else None
    except json.JSONDecodeError:
        return None

def locate(tokens, words, used):
    for st in range(len(tokens) - len(words) + 1):
        if tokens[st:st + len(words)] == words and (st, st + len(words)) not in used:
            return (st, st + len(words))
    return None

In [ ]:
conf, kinds, pair_words = Counter(), Counter(), {}
for p in preds:
    rec, q = gold[p["gold_idx"]], parse_json(p["raw"]) or {}
    tokens, used = rec["tokens"], set()
    gold_types = {(e["start"], e["end"]): e["type"] for e in rec["entities"]}
    for e in q.get("entities", []) if isinstance(q, dict) else []:
        span = locate(tokens, str(e.get("text", "")).split(), used)
        if not span:
            continue
        used.add(span)
        g_t, p_t = gold_types.get(span), str(e.get("type", ""))
        if g_t is None or g_t == p_t:
            continue
        conf[(g_t, p_t)] += 1
        pair_words.setdefault((g_t, p_t), Counter())[" ".join(tokens[span[0]:span[1]]).lower()] += 1
        g, pr = g_t.split("/"), p_t.split("/")
        kinds["different top-level class" if g[0] != pr[0] else "same class, different branch" if g[:2] != pr[:2]
              else "too general or too specific" if len(g) != len(pr) else "sibling subtype"] += 1
print("Kinds:", dict(kinds))
for (g_t, p_t), n in conf.most_common(15):
    print(f"{n:3d} | gold: {g_t}\n    | pred: {p_t}")

In [ ]:
train_word_types = {}
for r in train_rows:
    for e in json.loads(r["messages"][1]["content"])["entities"]:
        train_word_types.setdefault(e["text"].lower(), Counter())[e["type"]] += 1
all_gold_types = {e["type"] for r in gold for e in r["entities"]}
source, invented = Counter(), Counter()
for (g_t, p_t), words in pair_words.items():
    if p_t not in all_gold_types:
        invented[p_t] += sum(words.values())
    for word, k in words.items():
        tw = train_word_types.get(word, Counter()); total = sum(tw.values())
        source["word never seen in training" if total == 0 else "INHERITED from teacher" if tw[p_t] / total >= 0.5
               else "STUDENT ERROR" if tw[g_t] / total >= 0.5 else "mixed labels in training"] += k
total_err = sum(source.values())
for k, n in source.most_common():
    print(f"{n:4d} ({n / total_err:.0%})  {k}")
print("predictions using a type not in gold:", sum(invented.values()))